# PG-LIF — P1 Fix Sweep: selecting the v3 configuration on real SHD
**Context.** Telemetry established the failure mechanism on SHD: the tonic plateau drive κ·p is amplified ×20 by the somatic leak (steady state I/(1−αm)), pinning the population in a saturation–adaptation equilibrium (threshold ≈ 7), killing surrogate gradients (7–15% liveness) and exploding gradient norms (10¹²). However, the candidate fixes rank **differently** on synthetic proxies than the telemetry predicts for real data — the proxies are anti-predictive — so configuration selection must happen on SHD itself. That is what this notebook does, in one session (~45 min on a T4).

**Stage 1 (~25 min):** eight configurations, 5 epochs each, train and test accuracy:
1. `v2-base` — as previously run (control).
2. `scaled` — plateau drive κ·p·(1−αm), removing the ×20 tonic amplification (manuscript Eq. 9 form).
3. `beta0` — adaptation off (best 5-epoch variant in diagnostics: 21%).
4. `scaled+beta0` — both.
5. `highpass` — drive κ·(p − p̄) with a slow running mean p̄ (τ = 200 steps): removes the tonic component while keeping the informative fluctuations at full gain; interpretable as a slow homeostatic current opposing sustained depolarization.
6. `highpass+beta0.5` — high-pass with halved adaptation.
7. `taup15+beta0` — fast plateau with adaptation off.
8. `kappa0` — plateau disconnected (κ = 0 fixed): the harness ceiling without any plateau; every viable config must beat this control or the plateau is not paying for itself on this task.

**Stage 2 (~18 min):** the two best Stage-1 configs continue to 20 epochs for a stable reading.

Results (JSON + learning-curve figure) are saved to `My Drive/PG_LIF/P1_results/fix_sweep_<stamp>/`.

In [ ]:
import os, json, time, math
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT): ROOT = '/content/drive/MyDrive'
    BASE = os.path.join(ROOT, 'PG_LIF')
except Exception:
    BASE = './PG_LIF'
DATA = os.path.join(BASE, 'data', 'SHD')
OUT = os.path.join(BASE, 'P1_results', 'fix_sweep_' + time.strftime('%Y%m%d_%H%M%S'))
os.makedirs(OUT, exist_ok=True)
T_BINS, MAX_TIME, N_IN, N_OUT, HIDDEN, BATCH, LR = 100, 1.4, 700, 20, 128, 64, 5e-4
print('Sweep folder:', OUT)

In [ ]:
import numpy as np, h5py, torch, torch.nn as nn
def load_split(fname):
    with h5py.File(os.path.join(DATA, fname), 'r') as f:
        return ([np.array(t) for t in f['spikes']['times']],
                [np.array(u) for u in f['spikes']['units']],
                np.array(f['labels'], dtype=np.int64))
TR = load_split('shd_train.h5'); TE = load_split('shd_test.h5')
def batches(split, batch_size, shuffle, device='cpu'):
    times, units, labels = split
    idx = np.random.permutation(len(labels)) if shuffle else np.arange(len(labels))
    for b0 in range(0, len(idx), batch_size):
        sel = idx[b0:b0+batch_size]
        x = torch.zeros(len(sel), T_BINS, N_IN)
        for i, j in enumerate(sel):
            tt = times[j]; uu = units[j]; keep = tt < MAX_TIME
            tb = np.clip((tt[keep] / MAX_TIME * T_BINS).astype(int), 0, T_BINS-1)
            x[i, tb, uu[keep]] = 1.0
        yield x.to(device), torch.as_tensor(labels[sel]).to(device)
print('train', len(TR[2]), '| test', len(TE[2]))

In [ ]:
class Triangle(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x); return (x >= 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * torch.clamp(1.0 - x.abs(), min=0.0)
spike_fn = Triangle.apply
def decay(tau): return math.exp(-1.0 / tau)

class PGLIFCell(nn.Module):
    def __init__(self, N, beta=1.0, kappa0=1.0, tau_p=None, theta_d=1.0, tref_p=10,
                 scaled=False, highpass=False, kappa_fixed_zero=False):
        super().__init__(); self.N = N
        self.am = decay(20); self.ad = decay(20); self.aa = decay(200); self.ahp = decay(200)
        tau_p = tau_p or T_BINS / 2
        ap0 = decay(tau_p)
        self.ap_logit = nn.Parameter(torch.full((N,), math.log(ap0 / (1 - ap0))))
        self.kappa = nn.Parameter(torch.full((N,), float(kappa0)))
        self.beta, self.th, self.th_d, self.P0, self.tref_p = beta, 1.0, theta_d, 1.0, tref_p
        self.scaled, self.highpass, self.kzero = scaled, highpass, kappa_fixed_zero
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.vd, self.p, self.a, self.pbar = z(), z(), z(), z(), z()
        self.rp = torch.zeros(B, self.N, device=dev)
    def forward(self, I_ff, I_rec):
        self.vd = self.ad * self.vd + I_ff
        ed = spike_fn(self.vd - self.th_d) * (self.rp == 0).float()
        self.rp = torch.clamp(self.rp - 1, min=0) + ed.detach() * self.tref_p
        self.p = torch.sigmoid(self.ap_logit) * self.p + self.P0 * ed
        if self.kzero:
            drive = 0.0
        elif self.highpass:
            self.pbar = self.ahp * self.pbar + (1 - self.ahp) * self.p.detach()
            drive = self.kappa * (self.p - self.pbar)
        elif self.scaled:
            drive = self.kappa * self.p * (1 - self.am)
        else:
            drive = self.kappa * self.p
        self.vs = self.am * self.vs + I_ff + I_rec + drive
        th = self.th + self.beta * self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

class RecSNN(nn.Module):
    def __init__(self, **kw):
        super().__init__()
        self.w_in = nn.Linear(N_IN, HIDDEN); self.w_rec = nn.Linear(HIDDEN, HIDDEN, bias=False)
        self.cell = PGLIFCell(HIDDEN, **kw); self.w_out = nn.Linear(HIDDEN, N_OUT)
        self.a_out = decay(20); nn.init.orthogonal_(self.w_rec.weight)
    def forward(self, x):
        B, T, _ = x.shape; self.cell.init(B, x.device)
        s = torch.zeros(B, HIDDEN, device=x.device)
        out = torch.zeros(B, N_OUT, device=x.device); vo = torch.zeros(B, N_OUT, device=x.device)
        for t in range(T):
            s = self.cell(self.w_in(x[:, t]), self.w_rec(s))
            vo = self.a_out * vo + self.w_out(s); out = out + vo
        return out

def accuracy(model, split, device, limit=1024):
    model.eval(); correct = tot = 0
    with torch.no_grad():
        for x, y in batches(split, 256, shuffle=False, device=device):
            out = model(x); correct += (out.argmax(1) == y).sum().item(); tot += len(y)
            if tot >= limit: break
    return correct / tot

def train_cfg(tag, epochs, resume=None, **kw):
    torch.manual_seed(0); np.random.seed(0)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    m = RecSNN(**kw).to(device)
    o = torch.optim.Adam(m.parameters(), lr=LR); cr = nn.CrossEntropyLoss()
    hist = []
    if resume is not None:
        m.load_state_dict(torch.load(resume['ckpt'])); hist = resume['history']
        o.load_state_dict(torch.load(resume['opt']))
    for ep in range(len(hist), epochs):
        m.train(); t0 = time.time()
        for x, y in batches(TR, BATCH, shuffle=True, device=device):
            o.zero_grad(); cr(m(x), y).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0); o.step()
        hist.append({'epoch': ep, 'train_acc': accuracy(m, TR, device),
                     'test_acc': accuracy(m, TE, device)})
        print(f"{tag:18s} ep{ep:02d} train {hist[-1]['train_acc']:.3f} test {hist[-1]['test_acc']:.3f} {time.time()-t0:.0f}s")
    ck = os.path.join(OUT, tag.replace(' ', '_') + '.pt'); ok = ck.replace('.pt', '_opt.pt')
    torch.save(m.state_dict(), ck); torch.save(o.state_dict(), ok)
    return {'tag': tag, 'kw': {k: str(v) for k, v in kw.items()}, 'history': hist,
            'ckpt': ck, 'opt': ok}

## Stage 1 — eight configurations, 5 epochs

In [ ]:
CONFIGS = [
  ('v2-base',           dict()),
  ('scaled',            dict(scaled=True)),
  ('beta0',             dict(beta=0.0)),
  ('scaled+beta0',      dict(scaled=True, beta=0.0)),
  ('highpass',          dict(highpass=True)),
  ('highpass+beta0.5',  dict(highpass=True, beta=0.5)),
  ('taup15+beta0',      dict(tau_p=15, beta=0.0)),
  ('kappa0 (control)',  dict(kappa_fixed_zero=True)),
]
STAGE1 = [train_cfg(tag, 5, **kw) for tag, kw in CONFIGS]
json.dump([{k: r[k] for k in ('tag', 'kw', 'history')} for r in STAGE1],
          open(os.path.join(OUT, 'stage1.json'), 'w'), indent=2)
print()
for r in STAGE1:
    print(f"{r['tag']:20s} best test {max(h['test_acc'] for h in r['history']):.3f}  "
          f"final train {r['history'][-1]['train_acc']:.3f}")

## Stage 2 — top two continue to 20 epochs
The κ = 0 control is excluded from promotion (it is a reference line, not a candidate), but its Stage-1 curve stays in the final plot.

In [ ]:
cand = [r for r in STAGE1 if 'control' not in r['tag']]
top2 = sorted(cand, key=lambda r: -max(h['test_acc'] for h in r['history']))[:2]
print('promoted:', [r['tag'] for r in top2])
STAGE2 = []
for r in top2:
    kw = {}
    for k, v in r['kw'].items():
        kw[k] = (v == 'True') if v in ('True', 'False') else float(v)
    STAGE2.append(train_cfg(r['tag'], 20, resume=r, **kw))
json.dump([{k: r[k] for k in ('tag', 'kw', 'history')} for r in STAGE2],
          open(os.path.join(OUT, 'stage2.json'), 'w'), indent=2)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
for r in STAGE1:
    if r['tag'] in [s['tag'] for s in STAGE2]: continue
    plt.plot([h['epoch'] for h in r['history']], [h['test_acc'] for h in r['history']],
             '--', lw=1, label=r['tag'])
for r in STAGE2:
    plt.plot([h['epoch'] for h in r['history']], [h['test_acc'] for h in r['history']],
             lw=2, label=r['tag'] + ' (20 ep)')
plt.axhline(0.7513, color='k', ls=':', lw=1, label='TC-LIF quick-mode (20 ep)')
plt.xlabel('epoch'); plt.ylabel('SHD test accuracy'); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig_fix_sweep.png'), dpi=300); plt.show()
print()
for r in STAGE2:
    print(f"FINAL {r['tag']:20s} best test {max(h['test_acc'] for h in r['history']):.3f}  "
          f"final train {r['history'][-1]['train_acc']:.3f}")
best = max(STAGE2, key=lambda r: max(h['test_acc'] for h in r['history']))
k0 = max(h['test_acc'] for h in [r for r in STAGE1 if 'control' in r['tag']][0]['history'])
print(f"\nkappa0 control (5 ep): {k0:.3f} - any winning config must clearly beat this to justify the plateau.")
print(f"VERDICT: v3 configuration = {best['tag']} -> goes into P1 v3 for the full benchmark.")

### Reading the results
- If a fixed config approaches TC-LIF's 75% (dotted line), the mechanism is confirmed and v3 is settled.
- If the winner only matches the **κ = 0 control**, then on SHD the plateau is not contributing beyond a well-behaved dual-input ALIF — a legitimate, reportable finding that shifts the paper's weight to the learning-gate claim (P3), exactly as the plan's risk register anticipates.
- If everything stays below ~30%, the problem is deeper than drive scaling and the next step is architecture-level (e.g., somatic refractory, drive normalization), not hyperparameters.

Upload the executed notebook back; the winning configuration goes into P1 v3 and the manuscript's model section gets amended to match whatever the data selected.